# Qwen3.5-0.8B Continued Pre-Training (CPT) Pipeline
### Kaggle Dual GPU Training Notebook (Phase 1 — DDP + Speed Optimizations)

This notebook runs full-parameter Continued Pre-Training on **Qwen3.5-0.8B-Base** (~1B target tokens) with Kaggle Multi-GPU acceleration and the following speed optimizations:

| Optimization | Effect | Quality Impact |
|:---|:---|:---|
| **Liger Kernel** | Fused CE/RMSNorm/SwiGLU/RoPE → 40-60% VRAM savings | Zero |
| **SDPA Attention** | Memory-efficient attention on T4 | Zero |
| **torch.compile** | Kernel fusion → 1.3-1.5× throughput | Zero |
| **8-bit AdamW** | Saves ~4.5GB VRAM | Negligible |
| **seq_len=2048** | 2× tokens/step (enabled by Liger VRAM savings) | Improved |
| **micro_batch=2** | Higher GPU utilization | Zero |

**Dataset mixture:**
- 35% Stack v3 Code (`HuggingFaceCode/stack-v3-train`)
- 20% Stack v3 Documentation (`.md`, `.rst`, `README`)
- 20% The Vault (`Fsoft-AIC/the-vault-function`)
- 15% FineWeb-HQ (`epfml/FineWeb-HQ`)
- 10% OpenWebMath (`open-web-math/open-web-math`)

## 0. Directory Setup & Repository Working Path

In [ ]:
import os, sys
# Ensure working directory is set to QaptaanLM-0.75B
if os.path.exists("/kaggle/working/QaptaanLM-0.75B"):
    os.chdir("/kaggle/working/QaptaanLM-0.75B")
elif os.path.exists("QaptaanLM-0.75B"):
    os.chdir("QaptaanLM-0.75B")
print(f"✓ Current working directory: {os.getcwd()}")
sys.path.insert(0, os.getcwd())

## 1. Hardware & Multi-GPU Check

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory / (1024**3):.1f} GB, SM{props.major}{props.minor})")

## 2. Install Dependencies (Including Speed Optimizations)

In [ ]:
# Core dependencies + Liger Kernel for fused Triton kernels
!pip install -q --upgrade transformers datasets accelerate peft bitsandbytes \
    datasketch xxhash pyyaml rich huggingface_hub liger-kernel

## 3. Verify Liger Kernel Installation

In [ ]:
# Verify Liger Kernel is available and compatible
try:
    import liger_kernel
    print(f"✓ Liger Kernel v{liger_kernel.__version__} installed")
    from liger_kernel.transformers import apply_liger_kernel_to_qwen3
    print("✓ Qwen3/Qwen3.5 patching API available")
except ImportError as e:
    print(f"⚠ Liger Kernel not available: {e}")
    print("Training will work but will be slower and use more VRAM.")
    print("Install with: pip install liger-kernel>=0.3.0")

# Verify bitsandbytes for 8-bit optimizer
try:
    import bitsandbytes
    print(f"✓ bitsandbytes v{bitsandbytes.__version__} installed (8-bit AdamW available)")
except ImportError:
    print("⚠ bitsandbytes not available — will use standard AdamW")

## 4. Hugging Face Authentication

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✓ Logged in to Hugging Face")
except Exception as e:
    print(f"Manual login needed or secret not found: {e}")

## 5. Verify Base Model Architecture (Text-Only / Strip Vision)

In [ ]:
from transformers import AutoTokenizer, Qwen3_5ForCausalLM

model_id = "Qwen/Qwen3.5-0.8B-Base"
print(f"Loading {model_id} as text-only Qwen3_5ForCausalLM...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = Qwen3_5ForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto"
)

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Loaded text model with {total_params:,} parameters ({total_params/1e9:.2f}B)")

# Free this verification model — the trainer will load its own
del model
torch.cuda.empty_cache()

## 6. Prepare Training Shards

In [ ]:
!mkdir -p /kaggle/working/data
!mkdir -p /kaggle/working/checkpoints
!mkdir -p /kaggle/working/logs

!python scripts/03_process_data.py --output-dir /kaggle/working/data

## 7. Run CPT Training (Multi-GPU DDP + All Optimizations)

Uses `torchrun --nproc_per_node=2` for PyTorch DistributedDataParallel (DDP) across both Kaggle T4 GPUs.

**Active optimizations:**
- Liger Kernel: FusedLinearCrossEntropy + FusedRMSNorm + FusedSwiGLU + FusedRoPE
- SDPA attention (memory-efficient, works on T4 — FlashAttention-2 does NOT)
- torch.compile (kernel fusion, ~1.3-1.5× throughput)
- 8-bit AdamW (saves ~4.5GB VRAM)
- seq_len=2048 (enabled by Liger VRAM savings)
- micro_batch=2 (auto-configured, falls back to 1 if OOM)

In [ ]:
# Launch training with PyTorch Distributed Data Parallel (DDP) across 2x T4 GPUs
!torchrun --nproc_per_node=2 scripts/05_train_cpt.py --data-dir /kaggle/working/data

# If running on a single GPU runtime (e.g. P100), run instead:
# !python scripts/05_train_cpt.py --data-dir /kaggle/working/data

## 8. Evaluate Model & Generate Benchmark Report

In [ ]:
!python scripts/06_evaluate.py --compare --base Qwen/Qwen3.5-0.8B-Base --cpt /kaggle/working/checkpoints/final --output /kaggle/working/logs/comparison.json